# Cookie Cats A/B Test: Gate at Level 30 vs Level 40

**Question:** Does moving the first progress gate from level 30 (`gate_30`, control) to level 40 (`gate_40`, treatment) change player retention and engagement?

- **Primary metric:** `retention_7`
- **Secondary metrics:** `retention_1`, `sum_gamerounds`
- **Settings:** α = 0.05, two-sided tests, 95% confidence intervals, intention-to-treat (all randomized players)

## Part 1 – Setup
### 1.1 Configuration
Edit `DATA_PATH` if your file sits somewhere else in Google Drive.

In [ ]:
# ---- Edit this path if needed ----
DATA_PATH = "/content/drive/MyDrive/cookie_cats.csv"

# Experiment settings
CONTROL, TREATMENT = "gate_30", "gate_40"
ALPHA = 0.05          # significance level (two-sided)
CONF_LEVEL = 0.95     # confidence level for all intervals
SEED = 42             # seed for bootstrap and permutation steps
N_BOOT = 5_000        # bootstrap / permutation resamples

### 1.2 Imports and plot style

In [ ]:
import os
import glob
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from statsmodels.stats.proportion import (
    proportions_ztest,
    confint_proportions_2indep,
    proportion_effectsize,
)
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings("ignore", category=FutureWarning)

# Reproducibility: one seeded generator for all resampling
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

# Plot style
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.figsize": (10, 5),
    "figure.dpi": 100,
    "axes.titleweight": "bold",
    "axes.titlesize": 13,
})
PALETTE = {CONTROL: "#1f77b4", TREATMENT: "#ff7f0e"}

# Table display
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

print(f"pandas {pd.__version__} | numpy {np.__version__} | seaborn {sns.__version__}")
print(f"Seed = {SEED}, alpha = {ALPHA}, bootstrap resamples = {N_BOOT:,}")

### 1.3 Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### 1.4 Load the data

In [ ]:
def load_data(path):
    """Load the CSV; print a clear message and stop if the path is wrong."""
    if not os.path.isfile(path):
        print(f"ERROR: file not found at:\n   {path}")
        print("Check DATA_PATH in cell 1.1 (spelling, folder, and that Drive is mounted).")
        # Help locate the file: list CSVs in the top level of My Drive
        candidates = sorted(glob.glob("/content/drive/MyDrive/*.csv"))
        if candidates:
            print("\nCSV files found in My Drive:")
            for c in candidates[:20]:
                print("   ", c)
        raise FileNotFoundError(path)
    return pd.read_csv(path)

df = load_data(DATA_PATH)

print(f"Loaded {DATA_PATH}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")

# Quick check against the expected size
EXPECTED_SHAPE = (90_189, 5)
if df.shape == EXPECTED_SHAPE:
    print("Shape matches the expected 90,189 x 5.")
else:
    print(f"Warning: expected {EXPECTED_SHAPE}, got {df.shape}. Confirm this is the right file.")

df.head()